In [0]:
# File uploaded to /FileStore/tables/Northwind_Curbal/Categories.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Calendar.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Customers.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Employees.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Products.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Orders.csv
# File uploaded to /FileStore/tables/Northwind_Curbal/Suppliers.csv

## Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Load the data

### Products

In [0]:
products_path = "/FileStore/tables/Northwind_Curbal/Products.csv" 

products = spark.read.format('csv')\
                .option('header','True')\
                .option('inferSchema','True')\
                .load(products_path)

### Orders

In [0]:
orders_path = "/FileStore/tables/Northwind_Curbal/Orders.csv"
orders = spark.read.format('csv')\
                .option('header','True')\
                .option('inferSchema','True')\
                .load(orders_path)

### Customers

In [0]:
customers_path = "/FileStore/tables/Northwind_Curbal/Customers.csv"
customers = spark.read.format('csv')\
                .option('header','True')\
                .option('inferSchema','True')\
                .load(customers_path)

### Employees

In [0]:
employees_path = "/FileStore/tables/Northwind_Curbal/Employees.csv"

employees = spark.read.format('csv')\
                .option("header", 'True')\
                .option("inferSchema",'True')\
                .load(employees_path)

### Suppliers

In [0]:
suppliers_path = "/FileStore/tables/Northwind_Curbal/Suppliers.csv"
suppliers = spark.read.format("csv")\
                .option('header','True')\
                .option('inferSchema','True').load(suppliers_path)

## Questions

### 1. How many current products cost less than $20?

1. Apply filter conditions

In [0]:
products.filter((col('UnitPrice') < 20) & (col('Discontinued') == 'false')).display()

### 2. Which product is most expensive?

#### Method 1

In [0]:
# 1. Sort the Unit Price by descending order and take the first record using first() method
products.sort(desc('UnitPrice')).select('ProductName').first()

#### Method 2

In [0]:
# 1. Sort the Unit Price by descending order and limit the result to 1st row and display it.
products.sort(desc('UnitPrice')).limit(1).select('ProductName').display()

### 3. What is average unit price for our products

In [0]:
# 1. Simply take the average of Unit price column
products.agg(round(avg('UnitPrice'),2).alias('AvgPrice')).display()

### 4. How many products are above the average unit price?

In [0]:
# 1. save the avg unit price from the previous query in a variable
# 2. Use the variable result to filter.
avg_unit_price = products.agg(round(avg('UnitPrice'),2).alias('AvgPrice')).collect()[0]['AvgPrice']

products.filter(col('UnitPrice')>avg_unit_price).display()

In [0]:
products.filter(col('UnitPrice')>avg_unit_price).count()

### 5. How many products cost between $15 and $25 (inclusive)?

In [0]:
# 1. Use the filter condition on Unit Price to filter out the results
products.filter((col("UnitPrice") >=15) & (col('UnitPrice') <= 25)).display()

In [0]:
products.filter((col("UnitPrice") >=15) & (col('UnitPrice') <= 25)).count()


### 6. What is the average number of products (not qty) per order?

In [0]:
# 1. Select only the relevant columns (order_id, product_id)
# 2. Perform group by on order_id and count the number of product_id's per order_id
# 3. Sort the result by order_id in ascending order
# 4. Take the resulted query and perform average of the product count and display the result
orders.select('OrderID', 'ProductID')\
    .groupBy('OrderID').agg(count('ProductID').alias('ProductCount')).sort('OrderID')\
    .agg(round(avg('ProductCount'),2).alias('AvgProductCountPerOrder'))\
    .display()

### 7. What is Order Value in $ of open orders? (Not Shipped Yet)

In [0]:
# 1. Filter the columns where shipped date is null.
# 2. Add a new column "Total Sales" by multiplying the UnitPrice and Quantity column on row-level
# 3. Perform summation on the new column and display the result.
orders.filter(col('ShippedDate').isNull())\
    .withColumn('TotalSales', col('UnitPrice')*col("Quantity"))\
    .agg(round(sum('TotalSales'),2).alias('TotalSaleOrder'))\
    .display()

### 8. How many orders are 'single item' (only one product ordered)?

In [0]:
#1. Perform the group by on the "Order_id" and count the number of products per order
# 2. Filter out orders with only product and display the result
orders.groupBy('OrderID').agg(count('ProductID').alias('ProductCount'))\
    .filter(col('ProductCount')==1)\
    .display()

In [0]:
orders.groupBy('OrderID').agg(count('ProductID').alias('ProductCount'))\
    .filter(col('ProductCount')==1).count()

### 9. Average Sales per Transaction (orderID) for "Romero y Tomillo"

In [0]:
# 1. Filter the orders table where ship name is "Romero y tomillo"
# 2. Perform group by on the filtered table and apply summation on the "_Sales" column
# 3. We get total sales per every order
# 4. Perform Avg on the resulted query and display the result
orders.filter(col('ShipName')=="Romero y tomillo")\
    .groupBy('OrderID').agg(sum("_Sales").alias('sales_by_order'))\
    .agg(round(avg('sales_by_order'),2).alias('AvgSalesPerTransaction'))\
    .display()

### 10. How many days since "North/South" last purchase?

In [0]:
# 1. Filter the customers table where company name is "North/South" and take the customer_id and save it in a variable
# 2. Filter the orders table with the above customer id
# 3. Sort the filtered table with the order_date in descending order and limit it to the first order
# 4. Add "Current Date" column, which displays today's date
# 5. Perform date difference between last purchase date and current date, we will get the query result in number of days.
customer_id = customers.filter(col('CompanyName')=="North/South")\
                     .select("CustomerID").collect()[0]['CustomerID']

orders.filter(col('CustomerID') == customer_id)\
    .sort(desc(col("OrderDate"))).limit(1).select("OrderDate")\
    .withColumn('CurrentDate', current_date())\
    .withColumn('days', datediff(col('OrderDate'), col("CurrentDate")))\
    .display()

### 11. How many customers have ordered only once?

#### Method 1

In [0]:
# 1. GroupBy on Orders table with "Order_id", and "Customer_id" and count the number of customers
# 2. Again GroupBy over "Customer_id" and count the number of orders
# 3. Filter the data where number_of_orders is 1 and display the result
orders.groupBy('OrderID', 'CustomerID').agg(count('CustomerID'))\
    .groupBy('CustomerID').agg(count('CustomerID').alias('NumberOfOrders'))\
    .filter(col('NumberOfOrders')==1)\
    .display()

#### Method 2

In [0]:
# 1. GroupBy on orders table over "Customer_id"
# 2. Count the distinct order_id which gives the number of orders made by each customer
# 3. Filter the number of order to be 1 and display the result.
orders.groupBy('CustomerID').agg(countDistinct('OrderID').alias('NumberOfUniqueOrders'))\
    .filter(col("NumberofUniqueOrders")==1)\
    .display()

### 12. How many new customers in 2023?

In [0]:
# 1. 

window = Window.partitionBy('CustomerID').orderBy(col('OrderDate'))

orders.withColumn('FirstOrderDate', dense_rank().over(window))\
    .filter((col('FirstOrderDate')==1) & (year('OrderDate')==2023))\
    .select('CustomerID').distinct().count()

### 13. How many Lost Customers in 2023?

In [0]:
# 1. Create a Partition over "Customer_id" and order by "order_date" with window frame being unbounded preceding and unbounded following.

# 2. For Orders table, get the first order date and latest order date for every customer
# 3. Apply distinct to get the unique customers
# 4. Filter the data where latest order should be with in the 2023 and display the result
window = Window.partitionBy('CustomerID').orderBy('OrderDate').rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing)

orders.withColumn('first_order_date', first('OrderDate').over(window))\
    .withColumn('last_order_date', last("OrderDate").over(window))\
    .select('CustomerID', "first_order_date", "last_order_date").distinct()\
    .filter((col('last_order_date') <= "2023-12-31") & (col('last_order_date') >='2023-01-01'))\
.display()


### 14. How many customers have never purchased Queso Cabrales?

#### Method 1

In [0]:
# 1. Get the product_id from the products table where product name is "Queso Cabrales"
# 2. Filter the orders table for the same product_id and get the distinct customers list.
# 3. Perform a left_anti join on the customers list with customer_table and get the customers who never brought this product.
product_id = products.filter(col('ProductName') == "Queso Cabrales").select("ProductID").collect()[0]["ProductID"]

In [0]:
customers_with_queso = orders.filter(col("ProductID")==product_id).select(col("CustomerID")).distinct()

In [0]:
customers.join(customers_with_queso, 
               customers['CustomerID'] == customers_with_queso['CustomerID'], 'left_anti').display()

#### Method 2

In [0]:
# 1. Join Orders and Products table where product_name in products table is Queso Cabrales and get the distinct customers id list.
# 2. Perform a left_anti join  on the Customers with the resulted customers ids and get the customers who didn't brought the Queso Cabrales.
customers.join(orders.join(products.filter(col('ProductName')=="Queso Cabrales").select("ProductID"), 
            products['ProductID']==orders['ProductID'], 'inner').select("CustomerID").distinct(),
            on='CustomerID', how='left_anti').count()

### 15. How many customers have purchased only Queso Cabrales (per OrderID)?

In [0]:
# 1. find the customers who only brought one product in a given order
# 2. filter the orders table with product_id for Queso Cabrales
# 3. Join the customers and orders table and display the result.

Customers_with_one_product = orders.groupBy('CustomerID', 'OrderID').agg(count('ProductID').alias("ProductCount")).filter(col("ProductCount")==1)

orders_with_only_queso = orders.filter(col("ProductID") == product_id)

Customers_with_one_product.join(orders_with_only_queso, on='OrderID', how="inner")\
    .select(Customers_with_one_product['CustomerID']).display()


### 16. How many products are out of stock?

In [0]:
# 1. Filter the products table where Units in Stock is zero and discontinued is false and display the result
products.filter((col("UnitsInStock")==0) & (col("Discontinued")==False)).display()

### 17. How many products need to be restocked? (based on restock levels)

In [0]:
# 1. filter the products where units in stock is less than the reorder level.
products.filter(col("UnitsInStock") < col("ReorderLevel")).display()

### 18. How many products on order we need to restock?

In [0]:
# 1. Filter the products where unitsonorder is greater than unitsinstock and unitsonorder is more than zero.
products.filter((col("UnitsOnOrder") >= col("UnitsInStock")) & 
                (col("UnitsOnOrder") > 0)).display()

### 19. What is the stocked value of the discontinued products?

In [0]:
# 1. filter the products table where the discontinued is true
# 2. multiply the unit price and units in stock columns by row level
# 3. perform the summation on the resulted column and display the result.
products.filter(col("Discontinued")=='True')\
    .withColumn('Stock_Value', col("UnitPrice")*col("UnitsInstock"))\
    .agg(sum('Stock_Value').alias('TotalStockValue'))\
    .display()

### 20. Which vendor has the highest stock value?

In [0]:
# 1. multiply units in stock and unit price on products table and group by over the supplier id
# 2. perform summation over the resulted column and sort the supplier with highest stock value and fetch the result.

highest_supplier_id = products.withColumn("StockValue", col("UnitsInStock")*col("UnitPrice"))\
                .groupBy("SupplierID").agg(sum("StockValue").alias("StockValueBySupplier"))\
                .sort(desc("StockValueBySupplier")).select('SupplierID')\
                .limit(1).collect()[0]['SupplierID']


suppliers.filter(col("SupplierID") == highest_supplier_id).display()


### 21. How many employees are female?

In [0]:
# 1. Create a order by window on "gender" column with window frame being unbounded preceding and unbounded following

# 2. GroupBy employees column by "gender" 
window = Window.orderBy('gender').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
employees.groupBy('gender').agg(count('gender').alias('total'))\
    .withColumn("pct", round(100*(col('total')/sum(col("total")).over(window)),2))\
    .display()

In [0]:
# Perform aggregation to count the total number of employees by gender
gender_count = employees.groupBy('gender').agg(count('gender').alias('total'))

# Calculate the percentage in one step without a complex window function
total_count = gender_count.agg(sum('total').alias('total_sum')).collect()[0]['total_sum']

# Add a column to calculate the percentage for each gender
gender_pct = gender_count.withColumn("pct", F.round(100 * (col('total') / total_count), 2))

# Display the result
gender_pct.display()



### 22. How many employees are 60 years old or over?

In [0]:
# 1. create a new columns with today's date in it
# 2. calculate the difference between birth date and today'date, result will be number of days
# 3. Divide the resulted days with 365.25 to convert it to years (.25 is to take leap year into account)
# 4. Filter the rows where "age" is greater than 60 and fetch the result
employees.withColumn('today', current_date())\
    .withColumn('age', round((datediff(col('today'), col('BirthDate')))/365.25,2))\
    .filter(col("age") > 60)\
    .display()

In [0]:
employees.withColumn('today', current_date())\
    .withColumn('age', round((datediff(col('today'), col('BirthDate')))/365.25,2))\
    .filter(col("age") >= 60)\
    .count()

### 23. Which employee had the highest sales in 2022?

In [0]:
# 1. filter data where year of order_date is 2023
# 2. Perform groupBy over employee_id and do summation on the sales
# 3. sort the resulted df in descending order by sales and display the reuslt.
orders.filter(year(col("OrderDate")) == 2023)\
    .groupBy("EmployeeID").agg(round(sum("_Sales"),2).alias("SaleValue"))\
    .sort(desc("SaleValue"))\
    .display()

In [0]:
employees.join(orders.groupBy("EmployeeID", year(col("OrderDate")).alias("Year"))\
    .agg(round(sum("_Sales"),2).alias("SaleValue")),\
    on="EmployeeID", how='inner')\
    .select(orders["EmployeeID"], "Full Name", "Year", "SaleValue")\
    .filter(col("Year") == 2023)\
    .sort(desc("Year"),desc("SaleValue"))\
    .display()

### 24. How many employees sold over $100k in 2023?

In [0]:
# 1. Filter the year of order date to be 2023
# 2. Perform group by over employee_id and do summation over sales
# 3. Filter the result where sales is greater than 100000 and fetch the result 
orders.filter(year(col("OrderDate")) == 2023)\
    .groupBy("EmployeeID").agg(round(sum("_Sales"),2).alias("SaleValue"))\
    .sort(desc("SaleValue"))\
    .filter(col("SaleValue")>= 100000)\
    .display()

In [0]:
employees.join(orders.groupBy("EmployeeID", year(col("OrderDate")).alias("Year"))\
    .agg(round(sum("_Sales"),2).alias("SaleValue")),\
    on="EmployeeID", how='inner')\
    .select(orders["EmployeeID"], "Full Name", "Year", "SaleValue")\
    .filter(col("Year") == 2023)\
    .sort(desc("Year"),desc("SaleValue"))\
    .filter(col("SaleValue") > 100000)\
    .display()

### 25. How many employees got hired in 1994?

In [0]:
# 1. filter the data where year of hire date is 1994
employees.filter(year(col("HireDate"))==1994).display()